# Week 2 Day 1 — Evals & Observability

**What this notebook builds:** the evaluation harness for the UnderCover agent — Langfuse datasets, traced experiments, and business-grade scoring — plus a long-term memory the agent can search, so we can *measure* whether memory makes it a better claims handler.

The journey, in four parts:

1. **The even harder claims** — 25 claims that can't be decided from the policy wording alone. Each needs an outside fact (what a product is, when a storm was named) that lives in a small *knowledge base*. The handlers' decisions are our ground truth.
2. **A memory over MCP** — a Qdrant vector database run as an MCP server, giving the agent two tools: `qdrant-store` and `qdrant-find`. The agent decides when to search; retrieval is a tool, not a pipeline stage.
3. **Teaching the agent** — we store every knowledge-base fact in the memory and show the before/after on questions the assistant couldn't answer cold.
4. **Experiments in Langfuse** — two claims agents, one with just the payout tool and one that can also search the memory, run over a Langfuse dataset and scored claim by claim: accuracy, Cohen's kappa, the dollar cost of mistakes, and whether the needed facts were actually retrieved.

**Requirements:** `OPENROUTER_API_KEY` and Langfuse keys in `.env`, and `uvx` available to launch the Qdrant MCP server.

> ⚠️ **Note on the saved results:** the experiment outputs currently stored in this notebook are invalid — a variable-shadowing bug in the task function (see footnote **[^1]** in *Code improvements* at the end) made every claim fall back to `refer / $0.00`, so both experiments scored 8% accuracy. Fix the bug and re-run Part 4 to get a real comparison.


In [1]:
# Import necessary libraries

import json # For handling JSON data
import os # For Handling operating system operations
import shutil # For high-level file operations
from pathlib import Path # For handling filesystem paths
from typing import Literal # For type hinting with literal values

import httpx # For making HTTP requests
import pandas as pd # For data manipulation and analysis
import plotly.express as px # For creating interactive visualizations
from dotenv import load_dotenv # For loading environment variables from a .env file

from langchain.agents import create_agent # For creating langchain agents
from langchain.agents.middleware import ToolRetryMiddleware # For handling tool retry logic in langchain agent 
from langchain_core.tools import tool # For defining custom tools in langchain
from langchain_mcp_adapters.client import MultiServerMCPClient # For interacting with multiple MCP servers
from langchain_openai import ChatOpenAI # For interacting with OpenAI chat models

from langfuse import Evaluation, get_client # For interacting with Langfuse evaluation client
from langfuse.langchain import CallbackHandler # For handling Langfuse callbacks in Langchain

from week2day1_payout import Benefit, Line, Payout, work_out_payout

## Connect to OpenRouter and Langfuse

In [2]:
# Load keys from .env and check that OpenRouter and Langfuse are ready

load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
langfuse = get_client()

if openrouter_api_key:
    print("OpenRouter API Key is found and good to go")
else: 
    print("No OpenRouter API Key found")

if langfuse.auth_check():
    print("Connected to Langfuse")
else:
    print("Failed to connect to Langfuse")

OpenRouter API Key is found and good to go
Connected to Langfuse


## Part 1: The even harder claims

**Summary — Part 1.** We load the evaluation data from `data/evals/even_harder`: the claims, the handlers' decisions (`labels`), the customers' policies, and the full policy wording. The Tokyo lost-bag claim shows the shape of one claim, and the knowledge base holds the outside facts — the things a seasoned handler would just *know* — that make these claims "even harder". A claim whose label has `needs` entries can only be decided correctly if the agent finds those facts.


In [3]:
# Load the even harder claims, the handlers' decisions, the policies and the policy wording

EVEN_HARDER = Path("../../../data/evals/even_harder")

def load(name):
    """ Read one of the JSON files of the even harder claims."""
    return json.loads((EVEN_HARDER / name).read_text(encoding="utf-8"))

claims = load("claims.json")
labels = {decision['claim_id']: decision for decision in load("decisions.json")}
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = Path("../../../data/policy_wording.md").read_text(encoding="utf-8")

In [4]:
# One row per claim: the claim's summary and amount beside the handler's decision

rows = []
for claim in claims:
    label = labels[claim['claim_id']]
    rows.append({
        'claim_id': claim['claim_id'],
        'summary': claim['summary'],
        'amount_claimed': claim['amount_claimed'],
        'decision': label['decision'],
        'payout': label['payout']
    })

data = pd.DataFrame(rows)
data

,claim_id,summary,amount_claimed,decision,payout
0,CLM-1301,Bahamas trip canceled when Hurricane Marisol s...,1835.00,decline,0.00
1,CLM-1302,Cancun vacation canceled when Hurricane Tobias...,3197.00,approve,2473.00
2,CLM-1303,Stuck in Lisbon for a day by an air traffic co...,282.30,decline,0.00
3,CLM-1304,Flight to Denver canceled by the Meridian cabi...,734.40,approve,300.00
4,CLM-1305,"Ferry to Dublin canceled by Storm Ysolde, two ...",432.30,decline,0.00
5,CLM-1306,Train to Rome canceled by the national rail st...,605.60,approve,427.60
6,CLM-1307,Checked bag lost on the way to Tokyo,1611.95,approve,562.95
7,CLM-1308,Robbed outside a metro station in Barcelona,1329.00,approve,1239.00
8,CLM-1309,Backpack stolen on the Paris metro,771.99,approve,671.99
9,CLM-1310,Rented apartment in Split broken into,833.00,approve,434.00


### A lost bag in Tokyo

In [5]:
# One example claim: CLM-1307, a lost bag in Tokyo

tokyo = [claim for claim in claims if claim['claim_id'] == 'CLM-1307'][0]

print(tokyo['story'])
pd.DataFrame(tokyo['items'])

My checked bag never arrived in Tokyo. Skylark searched for a month, declared it lost and paid me $400. Everything in it is listed, with receipts for anything over $100.


,description,category,amount
0,Suitcase,luggage,189.00
1,Halden Arc,electronics,899.00
2,Lightweight rain jacket,clothing,165.00
3,Two merino T-shirts,clothing,118.00
4,"Hiking shoes, one pair",clothing,139.95
5,Packing cubes,luggage,54.20
6,Charging cables and adapters,electronics,46.80


In [6]:
# The handler's decision and payout calculation for the Tokyo claim

decision = labels['CLM-1307']

print(decision['reason'])
print(decision['calculation'])

The carrier declared the bag lost. The Halden Arc is a pocket projector, which is not on the Electronics Plus list, so it counts at the $300 per-item limit on Premium, and the other lines are within the limit. The items come to $1,012.95, less the carrier's $400 and the $50 deductible, leaving $562.95, inside the $1,500 maximum.
[{'step': 'Suitcase', 'amount': 189.0}, {'step': 'Halden Arc pocket projector, limited to $300.00', 'amount': 300.0}, {'step': 'Lightweight rain jacket', 'amount': 165.0}, {'step': 'Two merino T-shirts', 'amount': 118.0}, {'step': 'Hiking shoes', 'amount': 139.95}, {'step': 'Packing cubes', 'amount': 54.2}, {'step': 'Charging cables and adapters', 'amount': 46.8}, {'step': 'Less what others have paid or refunded', 'amount': -400}, {'step': 'Less the deductible', 'amount': -50.0}]


### The Knowledge Base

In [7]:
# The knowledge base: the outside facts the harder claims depend on

knowledge = json.loads(Path("../../../data/knowledge.json").read_text(encoding='utf-8'))

pd.set_option('display.max_colwidth', None)
pd.DataFrame(knowledge)

,topic,fact
0,events,"National Hurricane Center, July 9, 2026: Tropical Storm Marisol has formed east of the Leeward Islands. It is forecast to become a hurricane on July 11 and to pass over the Bahamas on July 12 and 13."
1,events,"National Hurricane Center, August 18, 2026: Tropical Storm Tobias has formed in the western Caribbean. It is expected to become a hurricane and to reach the coast of the Yucatan Peninsula near Cancun on August 21."
2,events,"Lisbon, June 20, 2026: Portugal's air traffic controllers have announced a 48 hour strike starting at midnight on July 14. Most flights to and from Lisbon and Porto are expected to be canceled on July 14 and 15."
3,events,"Meridian Air press release, August 5, 2026: The union representing our cabin crew has called a strike for August 12 and 13. We expect to cancel about half of our flights on those two days and will rebook affected passengers."
4,events,"North Atlantic weather bulletin, September 2, 2026: Storm Ysolde has been named. Gales are expected across Ireland, Wales and western Scotland from the morning of September 4, and ferry crossings on the Irish Sea are likely to be canceled."
5,events,"Rome, July 3, 2026: Italy's rail unions have called a 24 hour national rail strike from 9 pm on July 21 to 9 pm on July 22. Most high-speed and regional trains will not run."
6,events,"National Hurricane Center, June 30, 2026: Tropical Storm Nadia has formed in the southern Gulf of Mexico and is expected to come ashore in Tamaulipas, Mexico, on July 2."
7,events,"Northwind Airlines press release, July 15, 2026: The pilots' strike planned for July 24 has been called off after an agreement with the union. All flights will operate as scheduled."
8,events,"Japan Meteorological Agency, August 24, 2026: Typhoon Kaori is expected to pass close to Okinawa on August 27 and 28. Flights and ferries in the Ryukyu Islands are likely to be suspended."
9,events,"Athens, August 19, 2026: Greek ferry workers have called a 24 hour strike for August 27. Ferries between Piraeus and the islands will stay in port."


## Part 2: A memory over MCP
### Connecting to the memory server

**Summary — Part 2.** The memory is a Qdrant vector database, but the agent never sees Qdrant directly: it runs as an **MCP server** over stdio (`uvx mcp-server-qdrant`), and `MultiServerMCPClient` turns the server's capabilities into LangChain tools. The agent gets `qdrant-store` (remember a fact) and `qdrant-find` (semantic search). We wipe `qdrant_data` first so each session starts with a blank memory, then build a small assistant whose only job is to store and recall facts.


In [8]:
# Start fresh: wipe any Qdrant memory left over from a previous run

MEMORY = Path('qdrant_data').resolve()
shutil.rmtree(MEMORY, ignore_errors=True)

In [9]:
# Launch the Qdrant MCP server over stdio and list the memory tools it offers

MEMORY_ENV = {
    'QDRANT_LOCAL_PATH': str(MEMORY),
    'COLLECTION_NAME': 'memory'
}
parameters = {
    'transport': 'stdio',
    'command': 'uvx',
    'args': ['mcp-server-qdrant@0.8.1'],
    'env': MEMORY_ENV
}
client = MultiServerMCPClient({
    'memory': parameters
})
memory_tools = await client.get_tools()
for memory_tool in memory_tools:
    print(f"{memory_tool.name}: {memory_tool.description}\n")

qdrant-find: Look up memories in Qdrant. Use this tool when you need to: 
 - Find memories by their content 
 - Access memories for further analysis 
 - Get some personal information about the user

qdrant-store: Keep the memory for later use, when you are asked to remember something.



### An agent with memory

In [10]:
# A memory-only assistant: gpt-6-luna with the Qdrant store and find tools  [^3]

MEMORY_INSTRUCTIONS = """ you have a long-term memory.
When you are told a fact, store it in your long-term memory
word for word. When you are asked a a question, search your
memory and answer from what you find.
"""

http_client=httpx.AsyncClient(
    limits=httpx.Limits(max_keepalive_connections=0 )
)
luna = ChatOpenAI(
    model='openai/gpt-6-luna',
    base_url=OPENROUTER_BASE_URL,
    api_key=openrouter_api_key,
    reasoning_effort='low',
    timeout=120,
    max_retries=3,
    http_async_client=http_client)

assistant = create_agent(
    model=luna,
    tools=memory_tools,
    system_prompt=MEMORY_INSTRUCTIONS,
    middleware=[ToolRetryMiddleware(max_retries=3)])

In [11]:
# Before teaching: the assistant has an empty memory, so it shouldn't know these answers

async def ask(message):
    """ Send a message to the assistant in a new conversation and return its reply."""
    result = await assistant.ainvoke(
        {
            'messages': [
                {
                    'role': 'user',
                    'content': message
                }
            ]
        }
    )
    return result['messages'][-1].text
print(await ask("what kind of device is the Halden Arc?"))
print(await ask('When was Tropical Storm Marisol named?'))

I don’t have any information about the Halden Arc in my memory. What context did you hear the name in?
I couldn’t find information about Tropical Storm Marisol in my memory. Which year or region are you asking about?


## Part 3: Teaching the agent

**Summary — Part 3.** Teaching is just talking: we send each knowledge-base fact to the assistant with "Please remember this", and it stores the fact with `qdrant-store`. The before/after pair of questions is the proof — before teaching the assistant can't answer; after, it answers from memory. The direct `qdrant-find` query at the end shows *why* this works: embeddings match "a gadget for watching films on a wall" to the right fact with no keyword overlap. That's the semantic search at the heart of RAG.


In [12]:
# Teach the assistant: store every knowledge-base fact in its memory

for entry in knowledge:
    await ask(f"Please remember this: {entry['fact']}")
    print(f"Told: {entry['fact'][:80]}...")

Told: National Hurricane Center, July 9, 2026: Tropical Storm Marisol has formed east ...
Told: National Hurricane Center, August 18, 2026: Tropical Storm Tobias has formed in ...
Told: Lisbon, June 20, 2026: Portugal's air traffic controllers have announced a 48 ho...
Told: Meridian Air press release, August 5, 2026: The union representing our cabin cre...
Told: North Atlantic weather bulletin, September 2, 2026: Storm Ysolde has been named....
Told: Rome, July 3, 2026: Italy's rail unions have called a 24 hour national rail stri...
Told: National Hurricane Center, June 30, 2026: Tropical Storm Nadia has formed in the...
Told: Northwind Airlines press release, July 15, 2026: The pilots' strike planned for ...
Told: Japan Meteorological Agency, August 24, 2026: Typhoon Kaori is expected to pass ...
Told: Athens, August 19, 2026: Greek ferry workers have called a 24 hour strike for Au...
Told: Paris, May 29, 2026: French air traffic controllers have called a strike for Jun...
Told: Corv

In [13]:
# After teaching: the same questions are now answered from memory

print(await ask("What kind of device is the Halden Arc?"))
print(await ask("When was Tropical Storm Marisol named?"))

The Halden Arc is a pocket projector that can project an image up to 100 inches from a phone or laptop.
Tropical Storm Marisol was named on **July 9, 2026**.


In [14]:
# Under the hood: query the memory directly — semantic search finds the fact from a loose description

search = [memory_tool for memory_tool in memory_tools if memory_tool.name == 'qdrant-find'][0]

results = await search.ainvoke(
    {
        'query': 'a gadget for watching films on a wall'
    }
)
for line in json.loads(results[0]['text']):
    print(line)

Results for the query 'a gadget for watching films on a wall'
<entry><content>Halden Arc: a pocket projector that throws a picture of up to 100 inches from a phone or laptop. List price $899.</content><metadata></metadata></entry>
<entry><content>Vantor Halo: a titanium smart ring that tracks sleep, heart rate and activity. List price $449.</content><metadata></metadata></entry>
<entry><content>Lumora Drift 3: an e-reader with a 7 inch color E Ink screen and a battery that lasts six weeks. List price $329.</content><metadata></metadata></entry>
<entry><content>Corvane Lumen X1: the earlier Corvane smartwatch, released in 2024 at a list price of $449.</content><metadata></metadata></entry>
<entry><content>Lumora Drift 2: the earlier Lumora e-reader, with a 6 inch black and white screen. List price $199.</content><metadata></metadata></entry>
<entry><content>Halden Beam: a portable Bluetooth speaker with a built-in lamp. List price $199.</content><metadata></metadata></entry>
<entry><con

## Part 4: Experiments in Langfuse

**Summary — Part 4.** Now the evaluation itself. Two claims agents share the same instructions, policy wording, and tools — `record_decision` and `calculated_payout` — but only one can also search the memory (through a **read-only** client, so a confused agent can't overwrite facts). The claims become a Langfuse **dataset** with the handler's decision as the expected output; `run_experiment` runs an agent over every item `N_TIMES`, tracing each run in Langfuse, while our evaluators score it. Experiment 1 is the payout tool alone; Experiment 2 adds the memory; the final table compares them.


In [15]:
# The agent's instructions and the per-claim user prompt  [^6]

INSTRUCTIONS = """ 
You are a claims handler at UnderCover Travel Insurance. Decide on the 
claim you are given, using the policy wording below and the customer's 
policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are 
attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it because a required document is missing,
or the story and the documents disagree, or the wording does not settle the question.

Use your tools to record your decision.
"""

PAYOUT_TOOL_LINE = "Use the payout tool to work out the payout on an approved claim, and record the payout it returns.\n"

def make_user_prompt(claim):
    """ The customer's policy record and the claim as JSON. """
    policy = policies[claim['policy_id']]
    return f"""# Policy record
    {json.dumps(policy, indent=2)}
# Claim
    {json.dumps(claim, indent=2)}
"""

In [16]:
# The agent's tools: record a final decision, and work out a payout with UnderCover's rules  [^5]

recorded = {}

@tool(parse_docstring=True)
def record_decision(
    claim_id: str,
    rationale: str,
    decision: Literal['approve', 'decline', 'refer'],
    payout: float,
) -> str:
    """Record the decision on a claim in the claims system.

    Args:
        claim_id: The claim's id
        rationale: 2-3 sentences explaining the decision and relevant policy section
        decision: The decision on the claim
        payout: Amount to pay in US dollars, or 0 for a decline or referral
    """
    if claim_id in recorded:
        return f"A decision is already recorded on {claim_id} and is final."
    if decision == 'approve' and payout <= 0:
        return "An approval needs a payout above zero."

    recorded[claim_id] = {
        'decision': decision,
        'payout': payout,
        'rationale': rationale
    }
    return f"Decision {decision} recorded for {claim_id}."

@tool(parse_docstring=True)
def calculated_payout(policy_id: str,
                      benefit: Benefit,
                      lines: list[Line],
                      paid_by_others: float = 0.0) -> Payout:
    """ Work out the payout on an approved claim with UnderCover's payout rules.
    
    Args:
        policy_id: The customer's policy id
        benefit: The benefit the claim is paid under
        lines: The covered items and expenses, one for each line of the claim with any 
        excluded ones left out. A pair or set counts as one item
        paid_by_others: Anything a carrier, hotel, or other party has paid or refunded
        for the same loss
    
    Returns:
        The calculation, the payout and whether it needs sign-off
    """
    return work_out_payout(policies[policy_id], benefit, lines, paid_by_others)

### Two agents

In [17]:
# A read-only connection to the same memory, so the claims agents can search it but not change it

read_only_client = MultiServerMCPClient({
        'memory': {
            'transport': 'stdio',
            'command': 'uvx',
            'args':
            [
                'mcp-server-qdrant@0.8.1'
            ],
            'env':
            {
                **MEMORY_ENV,
                'QDRANT_READ_ONLY':'true'
            }
        }
    }
)

memory_search = await read_only_client.get_tools()
print([memory_tool.name for memory_tool in memory_search])


['qdrant-find']


In [18]:
# The two agents to compare: payout tool only, and payout tool plus memory search  [^4]

MEMORY_LINE = "Before you decide, search your memory for any product, event, or place" \
" in the claim that you need to know more about.\n"
payout_agent = create_agent(
    model=luna,
    tools=[record_decision,
           calculated_payout],
    system_prompt=f"{INSTRUCTIONS} {PAYOUT_TOOL_LINE}\n{policy_wording}\n")
memory_agent = create_agent(
    model=luna,
    tools=[record_decision,
           calculated_payout,
           *memory_search],
    system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}{MEMORY_LINE}\n{policy_wording}\n",
    middleware=[ToolRetryMiddleware(max_retries=3)],
)

### Scoring

**How the scoring works.** Each claim is scored on:

- **right** — the agent matched the handler's decision *and* the payout within a dollar.
- **mistake_cost** — mistakes in dollars, not counts: a wrongful decline costs \$1,500 (a complaint, maybe a regulator), a wrongful approval costs the payout itself, a needless referral \$50 of a colleague's time.
- **found_fact** — for claims that need a knowledge-base fact, whether the agent's memory searches actually retrieved it. This separates "couldn't find the fact" from "found it and still decided wrong".

Per run we add **decision accuracy**, **precision/recall** on approvals, **Cohen's kappa** (agreement with the handlers beyond chance), **straight-through rate** (claims decided without a human), and the **LLM cost**. `score_claim` scores one item in Langfuse, `score_run` scores a whole run, and `score_table` builds the claim-by-claim table we analyse locally.


In [19]:
# Scoring helpers: correctness, the dollar cost of mistakes, fact retrieval, kappa and the confusion matrix

DECISIONS = [
    'approve',
    'decline',
    'refer'
]
WRONGFUL_DECLINE = 1500
NEEDLESS_REFERRAL= 50

def is_right(answer, expected):
    """ True when the decision matches the handler's and the payout is within a dollar. """
    return answer['decision'] == expected['decision'] and abs(answer['payout'] - expected['payout']) <= 1

def mistake_cost(answer, expected):
    """ What the agent's answer costs UnderCover against the handler's decision."""
    if answer['decision'] == expected['decision']:
        return abs(answer['payout'] - expected['payout']) if answer['decision'] == 'approve' else 0
    if answer['decision'] =='decline':
        return WRONGFUL_DECLINE
    if answer['decision'] == 'approve':
        return answer['payout']
    return NEEDLESS_REFERRAL

def found_fact(answer, expected):
    """ For a claim that needs a a fact from the knowledge base, whether the agent's searches found it."""
    if not expected['needs']:
        return None
    return all(fact in answer['retrieved'] for fact in expected['needs'])

def kappa(table):
    """ Cohen's kappa for the agent's decisions against the handlers'."""
    observed = (table['agent'] == table['handler']).mean()
    agent_share = table['agent'].value_counts(normalize=True)
    handler_share = table['handler'].value_counts(normalize=True)
    expected = sum(agent_share.get(decision, 0) * handler_share.get(decision, 0) for decision in DECISIONS)
    return (observed-expected) / (1 - expected)

def show_confusion_matrix(table, title):
    """ Draw the confusion matrix, with the handler's decision on each row and the agent's in each column. """
    matrix = pd.crosstab(table['handler'], 
                         table['agent']).reindex(index=DECISIONS,
                                                  columns=DECISIONS, 
                                                  fill_value=0)
    fig = px.imshow(
        matrix, 
        text_auto=True,
        color_continuous_scale='Blues',
        labels={
            'x': 'Agent',
            'y': 'Handler',
            'color': 'Claims'
        },
        title=title)
    fig.show()


In [20]:
# Langfuse evaluators: score each claim, build a table per run, and score the whole run

def score_claim(*, output, expected_output, **kwargs):
    """ Item evaluator: whether the agent got the claim right, the cost of its mistake, and whether it found the fact it needed. """
    evaluations = [
        Evaluation(
            name='right',
            value = int(is_right(output,
                                 expected_output))),
        Evaluation(
            name='mistake_cost', 
            value=mistake_cost(output, 
                               expected_output)),
    ]
    found = found_fact(output, expected_output)
    if found is not None:
        evaluations.append(Evaluation(
            name='found_fact',
            value=int(found)
        ))
    return evaluations

def score_table(item_results):
    """ One row for each claim in a run: the agent's answer beside the handler's, with the scores. """
    rows = []
    for result in item_results:
        answer, expected = result.output, result.item.expected_output
        rows.append({
            'claim_id': result.item.id,
            'handler': expected['decision'],
            'agent': answer['decision'],
            'handler_payout': expected['payout'],
            'agent_payout': answer['payout'],
            'right': is_right(answer, expected),
            'mistake_cost': mistake_cost(answer, expected),
            'found_fact': found_fact(answer, expected),
            'llm_cost': answer['cost'],
            'rationale': answer['rationale']
        })
    return pd.DataFrame(rows).sort_values('claim_id')

def run_scores(table):
    """ The scores for a whole run. """
    agent_approvals = table[table['agent'] == 'approve']
    handler_approvals = table[table['handler'] == 'approve']
    return {
        'decision_accuracy': (table['agent'] == table['handler']).mean(),
        'accuracy': table['right'].mean(),
        'precision': (agent_approvals['handler'] == 'approve').mean(),
        'recall': (handler_approvals['agent'] == 'approve').mean(),
        'kappa': kappa(table),
        'cost_of_mistakes': table['mistake_cost'].sum(),
        'straight_through': (table['agent'] != 'refer').mean(),
        'found_fact': table['found_fact'].dropna().astype(float).mean(),
        'llm_cost': table['llm_cost'].sum()
    }

def score_run(*, item_results, **kwargs):
    """ Run evaluator: the scores for the whole run."""
    scores = run_scores(score_table(item_results))
    return [Evaluation(
        name=name,
        value=float(value)) for name, value in scores.items()]

def average_scores(table):
    """ The scores for each run of an experimental, averaged over the runs. """
    return pd.DataFrame([run_scores(run_table) for _, run_table in table.groupby('run')]).mean()

### The Dataset

In [21]:
# Create the Langfuse dataset, one item per claim with the handler's decision as the expected output  [^2]

DATASET = "even-harder-claims"

langfuse.create_dataset(
    name=DATASET,
    description="UnderCover's even harder claims with the handlers' decisions")

for claim in claims:
    label = labels[claim['claim_id']]
    langfuse.create_dataset_item(
        dataset_name=DATASET, 
        id=claim['claim_id'],
        input=claim,
        expected_output={
            'decision': label['decision'],
            'payout': label['payout'],
            'reason': label['reason'],
            'needs': label['needs'],
        # [^2] 'metadata' is nested inside expected_output — see Code improvements
        'metadata': {
            'tiny': label['tiny']
        }
        },
    )


In [22]:
# Choose what to run: all the claims, or just the tiny subset when TINY is True

TINY = False
N_TIMES = 3

dataset = langfuse.get_dataset(DATASET)
items = dataset.items
if TINY:
    items = [item for item in items if item.metadata['tiny']]
print(f"Running {len(items)} of the {len(dataset.items)} claims")

Running 25 of the 25 claims


In [23]:
# The task: run an agent on one claim, trace it in Langfuse, and collect its answer, cost and retrieved facts  [^1]

langfuse_handler = CallbackHandler()

async def decide(agent, claims):
    """ Run an agent on one claim and return its decision, the cost of its model calls and what its memory searches returned. """
    result = await agent.ainvoke(
        {
            'messages': [
                {
                    'role': 'user',
                    'content': make_user_prompt(claims)
                }
            ]
        },
        config={
            'callbacks': [langfuse_handler]
        },
    )
    # [^1] BUG: 'claim' is not the parameter (which is named 'claims') — this reads a leftover global
    answer = recorded.get(claim['claim_id'], 
                          {'decision':'refer',
                           'payout': 0.0,
                           'rationale': 'Nothing recorded'})
    cost = sum(message.response_metadata['token_usage']['cost'] for message in result['messages'] if message.type =='ai')
    retrieved = "\n".join(message.text for message in result['messages'] if message.type == 'tool' and message.name =='qdrant-find')
    return {**answer, 'cost': cost, 'retrieved': retrieved}

async def payout_task(*, item, **kwargs):
    """ Run the agent with the payout tool on one claim from the dataset. """
    return await decide(payout_agent, item.input)

async def memory_task(*, item, **kwargs):
    """ Run the agent with the payout tool and memory on one claim from the dataset. """
    return await decide(memory_agent, item.input)


### Experiment 1: The payout tool

In [24]:
# Experiment 1: the agent with the payout tool alone, N_TIMES over the dataset  [^7]

payout_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    result = langfuse.run_experiment(
        name='Payout tool',
        description='The agent with the payout tool',
        data=items,
        task=payout_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1
    )
    print(result.format())
    payout_tables.append(score_table(result.item_results). assign(run=run))

Individual Results: Hidden (25 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: Payout tool
📋 Run name: Payout tool - 2026-11-04T15:14:30.375921Z - The agent with the payout tool
25 items
Evaluations:
  • found_fact
  • mistake_cost
  • right

Average Scores:
  • found_fact: 0.000
  • mistake_cost: 46.000
  • right: 0.080

Run Evaluations:
  • decision_accuracy: 0.080
  • accuracy: 0.080
  • precision: nan
  • recall: 0.000
  • kappa: 0.000
  • cost_of_mistakes: 1150.000
  • straight_through: 0.000
  • found_fact: 0.000
  • llm_cost: 0.010

🔗 Dataset Run:
   https://us.cloud.langfuse.com/project/cmvs4vh500an7ad0my02nsrc4/datasets/cmw13npc804udad0ehdmakwrn/runs/9d66e58b-2706-42ef-9f1c-f485aab6c086
Individual Results: Hidden (25 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: Payout tool
📋 Run name: Payout tool - 2026-11-04T15:18:16.606134Z - The agen

In [25]:
# Stack the runs and draw the confusion matrix for the payout-tool agent

payout_table = pd.concat(payout_tables, ignore_index=True)
show_confusion_matrix(payout_table, 'Payout tool')

In [26]:
# How many claims the payout-tool agent got wrong

len(payout_table[~payout_table['right']])

69

### Experiment 2: The payout tool and memory

In [27]:
# Experiment 2: the agent with the payout tool and memory  [^7]

memory_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    result = langfuse.run_experiment(
        name='Payout tool and memory',
        description="The agent with the payout tool and memory of the knowledge base",
        data=items,
        task=memory_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
    )
    print(result.format())
    memory_tables.append(score_table(result.item_results).assign(run=run))

Individual Results: Hidden (25 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: Payout tool and memory
📋 Run name: Payout tool and memory - 2026-11-04T15:25:14.895939Z - The agent with the payout tool and memory of the knowledge base
25 items
Evaluations:
  • found_fact
  • mistake_cost
  • right

Average Scores:
  • found_fact: 1.000
  • mistake_cost: 46.000
  • right: 0.080

Run Evaluations:
  • decision_accuracy: 0.080
  • accuracy: 0.080
  • precision: nan
  • recall: 0.000
  • kappa: 0.000
  • cost_of_mistakes: 1150.000
  • straight_through: 0.000
  • found_fact: 1.000
  • llm_cost: 0.014

🔗 Dataset Run:
   https://us.cloud.langfuse.com/project/cmvs4vh500an7ad0my02nsrc4/datasets/cmw13npc804udad0ehdmakwrn/runs/b3dfac7c-99c2-451c-80d7-a60620bdc57b
Individual Results: Hidden (25 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: Payout tool and memor

In [28]:
# Stack the runs and draw the confusion matrix for the memory agent

memory_table = pd.concat(memory_tables, ignore_index=True)
show_confusion_matrix(memory_table, "Payout tool and memory")

In [29]:
# The claims the memory agent got wrong

memory_table[~memory_table['right']]

,claim_id,handler,agent,handler_payout,agent_payout,right,mistake_cost,found_fact,llm_cost,rationale,run
0,CLM-1301,decline,refer,0.00,0.0,False,50,True,0.000440,"Emergency Medical can cover necessary treatment of a sudden injury during the covered trip, and the incident falls within the trip dates. However, section 7 requires both an itemized bill and medical records; the submitted receipt is a total-only receipt, and the customer says the itemized bill is still forthcoming, so the required documentation is incomplete.",1
1,CLM-1302,approve,refer,2473.00,0.0,False,50,True,0.000699,"Emergency Medical can cover necessary treatment of a sudden injury during the covered trip, and the incident falls within the trip dates. However, section 7 requires both an itemized bill and medical records; the submitted receipt is a total-only receipt, and the customer says the itemized bill is still forthcoming, so the required documentation is incomplete.",1
2,CLM-1303,decline,refer,0.00,0.0,False,50,True,0.000396,"Emergency Medical can cover necessary treatment of a sudden injury during the covered trip, and the incident falls within the trip dates. However, section 7 requires both an itemized bill and medical records; the submitted receipt is a total-only receipt, and the customer says the itemized bill is still forthcoming, so the required documentation is incomplete.",1
3,CLM-1304,approve,refer,300.00,0.0,False,50,True,0.000495,"Emergency Medical can cover necessary treatment of a sudden injury during the covered trip, and the incident falls within the trip dates. However, section 7 requires both an itemized bill and medical records; the submitted receipt is a total-only receipt, and the customer says the itemized bill is still forthcoming, so the required documentation is incomplete.",1
4,CLM-1305,decline,refer,0.00,0.0,False,50,True,0.000473,"Emergency Medical can cover necessary treatment of a sudden injury during the covered trip, and the incident falls within the trip dates. However, section 7 requires both an itemized bill and medical records; the submitted receipt is a total-only receipt, and the customer says the itemized bill is still forthcoming, so the required documentation is incomplete.",1
...,...,...,...,...,...,...,...,...,...,...,...
68,CLM-1319,approve,refer,432.40,0.0,False,50,True,0.000471,"The fall and treatment occurred during the covered trip, and the records describe a sudden injury, but section 7 requires both an itemized bill and medical records. The submitted receipt only confirms the total ($451.50), and the story says the itemized bill is still forthcoming, so the required documentation is missing; request it before deciding.",3
69,CLM-1320,approve,refer,663.67,0.0,False,50,None,0.000879,"The fall and treatment occurred during the covered trip, and the records describe a sudden injury, but section 7 requires both an itemized bill and medical records. The submitted receipt only confirms the total ($451.50), and the story says the itemized bill is still forthcoming, so the required documentation is missing; request it before deciding.",3
70,CLM-1321,approve,refer,2875.80,0.0,False,50,None,0.000510,"The fall and treatment occurred during the covered trip, and the records describe a sudden injury, but section 7 requires both an itemized bill and medical records. The submitted receipt only confirms the total ($451.50), and the story says the itemized bill is still forthcoming, so the required documentation is missing; request it before deciding.",3
71,CLM-1322,approve,refer,300.00,0.0,False,50,None,0.000438,"The fall and treatment occurred during the covered trip, and the records describe a sudden injury, but section 7 requires both an itemized bill and medical records. The submitted receipt only confirms the total ($451.50), and the story says the itemized bill is still forthcoming, so the required documentation is missing; request it before deciding.",3


### Comparing the experiments

In [30]:
# The average scores for the two experiments, side by side

pd.DataFrame([average_scores(payout_table), average_scores(memory_table)], index=['Payout tool', 'Payout tool and memory']).round(3)

,decision_accuracy,accuracy,precision,recall,kappa,cost_of_mistakes,straight_through,found_fact,llm_cost
Payout tool,0.08,0.08,NaN,0.0,0.0,1150.0,0.0,0.0,0.008
Payout tool and memory,0.08,0.08,NaN,0.0,0.0,1150.0,0.0,1.0,0.012


## Conclusion

This notebook put the two pillars of Week 2 Day 1 in place:

- **Evals.** The handlers' decisions became a Langfuse dataset, and `run_experiment` turned "does memory help?" into a scored, repeatable comparison. The metrics are business metrics — dollars of mistakes and straight-through rate, not just accuracy — which is what makes an eval persuasive to UnderCover rather than just to engineers.
- **Observability.** Every agent run is traced in Langfuse through the `CallbackHandler`: each model call, tool call and token cost is inspectable after the fact. When something looks wrong in the score table, the trace shows *why*.
- **Memory as a tool over MCP.** The Qdrant server gave the agent `qdrant-store`/`qdrant-find`, a read-only flag made it safe to share, and the `found_fact` metric closed the loop by measuring whether searches surfaced what each claim needed.

**The caveat:** the saved comparison table is not evidence of anything yet — the `[^1]` bug meant neither agent's real decisions were ever scored (note `found_fact` = 1.0 for the memory agent, which shows the *searches* worked even while the scoring didn't). Fix `decide`, re-run Part 4, and then read the table. The eval harness doing its job is exactly this: an 8% accuracy score was the signal that something was broken.


## Code improvements (footnotes)

Suggestions only — no code has been changed. The `[^n]` markers appear as comments in the cells they refer to.

**[^1] CRITICAL — `decide` never reads the agent's recorded decision.** The parameter is named `claims`, but the lookup is `recorded.get(claim['claim_id'], ...)` — `claim` is **not the parameter**; Python resolves it to the leftover global from the dataset cell's `for claim in claims:` loop, so every item looks up the *same* claim id and falls back to `refer / $0.00 / "Nothing recorded"`. That is why both experiments scored 8% accuracy (just the handlers' own refers), precision is `NaN` (no approvals), straight-through is 0, and the confusion matrices are a single `refer` column. **Fix:** rename the parameter to `claim` (and pass `make_user_prompt(claim)`), then re-run Part 4. ~\$0.02 of the LLM spend bought the lesson that evals catch bugs too.

**[^2] `metadata` nested inside `expected_output` (dataset cell).** `'metadata': {'tiny': ...}` sits *inside* the `expected_output` dict instead of being a separate `metadata=` argument to `create_dataset_item`. It works while `TINY = False`, but `TINY = True` will crash (`item.metadata` is `None`), and the expected output carries a stray key. After fixing, re-run the dataset cell — Langfuse upserts items by id, so the stored items need repairing too.

**[^3] Memory instructions lost their escape hatch.** Compared to the lab, `MEMORY_INSTRUCTIONS` drops the line "If your memory has nothing on it, say that you don't know." Without it, the model may answer from pretraining and mask retrieval failures — which is exactly what the before/after demo is supposed to expose. Also: "asked a a question" and the lowercase opening.

**[^4] Uneven retry middleware.** `memory_agent` gets `ToolRetryMiddleware(max_retries=3)` but `payout_agent` gets none, so a flaky tool call punishes one side of the comparison. Also a stray space in `f"{INSTRUCTIONS} {PAYOUT_TOOL_LINE}"` (harmless).

**[^5] Tool name `calculated_payout`.** Tool names are part of what the model reads; a verb — `calculate_payout` — tells it what the tool *does* rather than what it did.

**[^6] Prompt formatting in `make_user_prompt`.** The f-string's source indentation puts four leading spaces before each JSON block, which Markdown can read as a code block — tidy, but worth knowing it's there. Consider dedenting.

**[^7] Duplicated experiment loop.** Experiments 1 and 2 are the same loop with different names and tasks; a helper like `run_n_times(name, description, task)` removes the duplication. Also a stray space in `score_table(result.item_results). assign(run=run)` — legal Python, but it reads like a typo.

**Bonus small things:**
- The `Failed to export spans ... Read timed out` warnings during Experiment 2 are transient Langfuse network flushes — they don't affect results; ignore or raise the client timeout.
- `N_TIMES = 3` × 25 claims × 2 experiments = 150 agent runs per full re-run — worth remembering when re-running after the `[^1]` fix (drop to 1 or set `TINY = True` first… after fixing `[^2]`).
- Spacing nits: `NEEDLESS_REFERRAL= 50`, docstring "an experimental" → "an experiment". A pass with `ruff format` would normalise these.
